# Analise dos sedimentos - Clarks Hill Lake

Este notebook analisa a planilha corrigida `Most Corrected Master_Data Clarks Hill Lake (version 3).xlsb.xlsx`, com foco em textura sedimentar, enriquecimento geoquimico e acoplamento entre ferro e carbono.

Principais perguntas:

- O sistema e dominado por sedimentos finos?
- A fracao fina controla o enriquecimento de ferro?
- O ferro esta associado ao carbono total?
- Quais sitios se comportam como zonas deposicionais de baixa energia?


In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd

MPLCONFIGDIR = Path(".matplotlib_cache")
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR.resolve()))

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.style.use("default")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

DATA_FILE = Path("Most Corrected Master_Data Clarks Hill Lake (version 3).xlsb.xlsx")
SHEET_NAME = "Master Data"

# Set to True if you want PNG copies of the figures.
SAVE_FIGS = False
FIG_DIR = Path("sediment_analysis_figures")

if SAVE_FIGS:
    FIG_DIR.mkdir(exist_ok=True)

def maybe_save(fig, name):
    if SAVE_FIGS:
        fig.savefig(FIG_DIR / name, dpi=220, bbox_inches="tight")


## 1. Leitura e limpeza dos dados

A aba `Master Data` contem os 19 sitios validos, alem de algumas linhas auxiliares no fim da tabela. A limpeza abaixo converte colunas numericas, remove linhas auxiliares e padroniza nomes como `Longtitude` para `Longitude` e `Ph` para `pH`.

In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Arquivo nao encontrado: {DATA_FILE.resolve()}")

raw = pd.read_excel(DATA_FILE, sheet_name=SHEET_NAME, engine="openpyxl")
raw.columns = [str(c).strip() for c in raw.columns]

df = raw.copy()

# Convert every column that can be numeric. Text-only helper rows become NaN in numeric fields.
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.rename(columns={"Longtitude": "Longitude", "Ph": "pH"})

# Keep sampled sites only. The workbook has extra helper/calibration rows after the main samples.
df = df[df["Site"].notna() & df["Fe_ppm"].notna()].copy()
df = df[(df["Site"] >= 1) & (df["Site"] <= 30)].copy()
df["Site"] = df["Site"].astype(int)
df = df.sort_values("Site").reset_index(drop=True)

print(f"Amostras validas: {len(df)}")
display(df.head())


In [ ]:
key_cols = [
    "Site", "Depth", "Latitude", "Longitude",
    "%Clay", "%Silt", "%Sand", "Water_%",
    "D10", "D50", "D90",
    "Fe_ppm", "Mn_ppm", "%Carbon", "pH", "DO (%)", "Cond."
]

available_key_cols = [c for c in key_cols if c in df.columns]
display(df[available_key_cols])


## 2. Estatistica descritiva

Esta tabela resume as variaveis sedimentologicas e geoquimicas mais importantes. O coeficiente de variacao ajuda a identificar quais variaveis mudam mais entre os sitios.

In [ ]:
summary_cols = [
    "%Clay", "%Silt", "%Sand", "Water_%", "D10", "D50", "D90",
    "Fe_ppm", "Mn_ppm", "%Carbon", "pH", "DO (%)", "Cond.", "Depth"
]
summary_cols = [c for c in summary_cols if c in df.columns]

summary = df[summary_cols].describe().T[["count", "mean", "std", "min", "50%", "max"]]
summary["cv_%"] = 100 * summary["std"] / summary["mean"]
display(summary.round(3))


## 3. Controle de qualidade rapido

Para granulometria, espera-se `D10 <= D50 <= D90`. Qualquer quebra nessa ordem deve ser conferida antes de interpretar `D90`.

In [ ]:
qa_grain = df[(df["D10"] > df["D50"]) | (df["D50"] > df["D90"])]

if qa_grain.empty:
    print("Nenhum problema D10/D50/D90 encontrado.")
else:
    print("Verificar estes valores granulometricos:")
    display(qa_grain[["Site", "D10", "D50", "D90", "%Clay", "%Silt", "%Sand"]])


## 4. Composicao granulometrica

O grafico de barras empilhadas mostra o dominio de argila + silte e a baixa participacao de areia na maioria dos sitios.

In [ ]:
texture_cols = ["%Clay", "%Silt", "%Sand"]

fig, ax = plt.subplots(figsize=(12, 5))
texture = df.set_index("Site")[texture_cols]
texture.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=["#5B8C5A", "#D6A84F", "#B86B4B"],
    edgecolor="white",
    linewidth=0.6,
)
ax.set_title("Composicao granulometrica por sitio")
ax.set_xlabel("Site")
ax.set_ylabel("Percentual (%)")
ax.legend(title="Fracao", ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.14))
ax.set_ylim(0, 105)
fig.tight_layout()
maybe_save(fig, "01_grain_size_stacked_bar.png")
plt.show()


In [ ]:
hist_cols = ["%Clay", "%Silt", "%Sand", "Water_%", "D50", "Fe_ppm", "Mn_ppm", "%Carbon"]
hist_cols = [c for c in hist_cols if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(14, 7), constrained_layout=True)
axes = axes.ravel()

for ax, col in zip(axes, hist_cols):
    values = df[col].dropna()
    ax.hist(values, bins=8, color="#4F6D7A", edgecolor="white", alpha=0.9)
    ax.axvline(values.mean(), color="#C44536", linestyle="--", linewidth=1.5, label="media")
    ax.axvline(values.median(), color="#1B998B", linestyle=":", linewidth=1.8, label="mediana")
    ax.set_title(col)
    ax.set_ylabel("Frequencia")
    ax.legend(fontsize=8)

for ax in axes[len(hist_cols):]:
    ax.axis("off")

fig.suptitle("Distribuicao das principais variaveis", y=1.03, fontsize=14)
maybe_save(fig, "02_distributions.png")
plt.show()


## 5. Correlacoes

A matriz abaixo ajuda a separar controles texturais, geoquimicos e hidrologicos. A leitura principal deve focar em sinal e magnitude, nao em significancia formal, porque `n = 19` e relativamente pequeno.

In [ ]:
corr_cols = [
    "Depth", "%Clay", "%Silt", "%Sand", "Water_%", "D10", "D50", "D90",
    "Fe_ppm", "Mn_ppm", "%Carbon", "pH", "DO (%)", "Cond."
]
corr_cols = [c for c in corr_cols if c in df.columns]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticks(range(len(corr_cols)))
ax.set_yticklabels(corr_cols)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        value = corr.iloc[i, j]
        if np.isfinite(value):
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=ax, shrink=0.8, label="r de Pearson")
ax.set_title("Matriz de correlacao")
fig.tight_layout()
maybe_save(fig, "03_correlation_matrix.png")
plt.show()

display(corr.round(3))


In [ ]:
selected_pairs = [
    ("%Clay", "Fe_ppm"),
    ("%Silt", "Fe_ppm"),
    ("%Sand", "Fe_ppm"),
    ("D50", "Fe_ppm"),
    ("Fe_ppm", "%Carbon"),
    ("%Clay", "%Carbon"),
    ("Water_%", "Fe_ppm"),
    ("Water_%", "%Carbon"),
    ("Fe_ppm", "Mn_ppm"),
    ("Depth", "Fe_ppm"),
    ("Depth", "D50"),
    ("Cond.", "Fe_ppm"),
]

rows = []
for x, y in selected_pairs:
    clean = df[[x, y]].dropna()
    if len(clean) < 2:
        continue
    r = clean[x].corr(clean[y])
    rows.append({"x": x, "y": y, "n": len(clean), "r": r, "R2": r**2})

selected_corr = pd.DataFrame(rows).sort_values("R2", ascending=False)
display(selected_corr.round(4))


## 6. Graficos de relacao com ajuste linear

Estes graficos resumem os controles principais: textura fina, tamanho mediano de grao, ferro, manganes, carbono, agua e profundidade.

In [ ]:
def scatter_reg(ax, data, x, y, color="#2E86AB"):
    clean = data[["Site", x, y]].dropna().copy()
    ax.scatter(clean[x], clean[y], s=70, color=color, edgecolor="black", linewidth=0.7, alpha=0.9)

    for _, row in clean.iterrows():
        ax.annotate(str(int(row["Site"])), (row[x], row[y]), xytext=(4, 4), textcoords="offset points", fontsize=8)

    if len(clean) >= 2 and clean[x].nunique() > 1:
        slope, intercept = np.polyfit(clean[x], clean[y], 1)
        xs = np.linspace(clean[x].min(), clean[x].max(), 100)
        ax.plot(xs, slope * xs + intercept, color="#C44536", linewidth=2)
        r = clean[x].corr(clean[y])
        ax.text(
            0.03, 0.95, f"n={len(clean)}\nr={r:.3f}\nR2={r*r:.3f}",
            transform=ax.transAxes, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#999999", alpha=0.9),
            fontsize=9,
        )

    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f"{x} vs {y}")

plot_pairs = [
    ("%Clay", "Fe_ppm"),
    ("D50", "Fe_ppm"),
    ("Fe_ppm", "%Carbon"),
    ("%Clay", "%Carbon"),
    ("Water_%", "%Carbon"),
    ("Depth", "Fe_ppm"),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
axes = axes.ravel()

for ax, (x, y) in zip(axes, plot_pairs):
    scatter_reg(ax, df, x, y)

fig.suptitle("Relacoes sedimentologicas e geoquimicas principais", y=1.03, fontsize=14)
maybe_save(fig, "04_key_scatter_regressions.png")
plt.show()


## 7. Padroes espaciais simples

Sem usar basemap, estes mapas em latitude/longitude permitem enxergar agrupamentos espaciais relativos. Os numeros indicam os sitios.

In [ ]:
map_cols = [("Fe_ppm", "YlOrRd"), ("%Carbon", "Greens"), ("D50", "Blues"), ("%Clay", "viridis")]

fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
axes = axes.ravel()

for ax, (col, cmap) in zip(axes, map_cols):
    clean = df[["Site", "Latitude", "Longitude", col]].dropna()
    sc = ax.scatter(
        clean["Longitude"], clean["Latitude"], c=clean[col], cmap=cmap,
        s=100, edgecolor="black", linewidth=0.8
    )
    for _, row in clean.iterrows():
        ax.annotate(str(int(row["Site"])), (row["Longitude"], row["Latitude"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.set_title(col)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    fig.colorbar(sc, ax=ax, label=col)

fig.suptitle("Distribuicao espacial relativa", y=1.03, fontsize=14)
maybe_save(fig, "05_spatial_patterns.png")
plt.show()


## 8. Indice deposicional fino-enriquecido

Este indice e uma composicao exploratoria, nao uma metrica padronizada da literatura. Ele aumenta com argila, agua, ferro e carbono, e diminui com `D50` e areia. Valores altos sugerem sitios mais finos, umidos e enriquecidos em Fe-C, consistentes com zonas deposicionais de menor energia.

In [ ]:
def zscore(series):
    return (series - series.mean()) / series.std(ddof=1)

df["Fine_depositional_score"] = (
    zscore(df["%Clay"]) +
    zscore(df["Water_%"]) +
    zscore(df["Fe_ppm"]) +
    zscore(df["%Carbon"]) -
    zscore(df["D50"]) -
    zscore(df["%Sand"])
)

score_table = df[[
    "Site", "Fine_depositional_score", "%Clay", "%Silt", "%Sand", "D50", "Water_%", "Fe_ppm", "%Carbon", "Depth"
]].sort_values("Fine_depositional_score", ascending=False)

display(score_table.round(3))


In [ ]:
score_sorted = df.sort_values("Fine_depositional_score", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

colors = np.where(score_sorted["Fine_depositional_score"] >= 0, "#2E8B57", "#B86B4B")
axes[0].bar(score_sorted["Site"].astype(str), score_sorted["Fine_depositional_score"], color=colors, edgecolor="black", linewidth=0.6)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Indice deposicional fino-enriquecido")
axes[0].set_xlabel("Site")
axes[0].set_ylabel("Score composto (z-scores)")

sc = axes[1].scatter(
    df["Longitude"], df["Latitude"], c=df["Fine_depositional_score"], cmap="RdYlGn",
    s=110, edgecolor="black", linewidth=0.8
)
for _, row in df.iterrows():
    axes[1].annotate(str(int(row["Site"])), (row["Longitude"], row["Latitude"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
axes[1].set_title("Score no espaco")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
fig.colorbar(sc, ax=axes[1], label="Fine depositional score")

maybe_save(fig, "06_fine_depositional_score.png")
plt.show()


## 9. Interpretacao sedimentologica

Com base nos dados, a interpretacao principal e:

- O sistema e dominado por sedimentos finos, principalmente argila + silte, com areia geralmente baixa.
- O ferro acompanha a fracao fina: aumenta com `%Clay` e diminui com `D50`, `%Silt` e `%Sand`.
- O carbono total esta fortemente associado ao ferro, sugerindo acoplamento organo-mineral e maior preservacao de carbono em sedimentos ricos em Fe.
- Profundidade maior tende a coincidir com sedimentos mais finos e enriquecidos em Fe, reforcando a interpretacao de zonas deposicionais de baixa energia.
- Sitios com maior score fino-enriquecido sao os melhores candidatos a zonas deposicionais ricas em Fe-C; sitios com score baixo representam condicoes relativamente mais grossas, menos umidas ou menos enriquecidas.

Ressalva: conferir manualmente valores granulometricos em que `D90 < D50`, especialmente antes de usar `D90` em conclusoes finais.